# Table of Contents

##### [1. Why Use a PDF Extraction Tool?](#1.-Why-Use-a-PDF-Extraction-Tool?)
##### [2. PDF Extraction Tooling Decisions](#2.-PDF-Extraction-Tooling-Decisions)
##### [3. Let's extract and reconstruct a sample report containing text, image and table.](#3.-Let's-extract-and-reconstruct-a-sample-report-containing-text,-image-and-table.)
  <!-- - [3.1 Load the PDF using PymuPDF](#3.1-Load-the-PDF-using-PymuPDF)
  - [3.2 Extract all text and visualize along with y coordinate](#3.2-Extract-all-text-and-visualize-along-with-y-coordinate)
  - [3.3 Extract all images and visualize along with y coordinate.](#3.3-Extract-all-images-and-visualize-along-with-y-coordinate.)
  - [3.4 Extract all tables and visualize along with y coordinate](#3.4-Extract-all-tables-and-visualize-along-with-y-coordinate)
  - [3.5 Let's reconstruct and visualize. We will have a text-table duplication problem here](#3.5-Let's-reconstruct-and-visualize.-We-will-have-a-text-table-duplication-problem-here) -->
##### [4. How to overcome Table and Text Overlap?](#4.-How-to-overcome-Table-and-Text-Overlap?)


## 1. Why use a PDF Extraction Tool?

A PDF extraction tool reads a PDF file and pulls out text, images, and tables.  
These tools preserve the x, y coordinates so they can be reconstructed

## 2. PDF Extraction Tooling Decisions

🔹 PyMuPDF (fitz) — Fast, Reliable for Text and Images. But fails for tables  
🔹 pdfplumber — for tables, reliable than pymupdf, 10 times faster than unstructred  
🔹 Unstructured — High accuracy but much slower and expensive   

🔹 Final Approach — Hybrid Pipeline  
PyMuPDF → Text and Images (Then pass Images to LLM later)   
pdfplumber → Tables    

Reference:  
https://onlyoneaman.medium.com/i-tested-7-python-pdf-extractors-so-you-dont-have-to-2025-edition-c88013922257

## 3. Let's extract and reconstruct a sample report containing text, image and table.

Preview

<img src="assets/sample_weather_report.png" height="400">

### 3.1 Load the PDF using PymuPDF

In [19]:
import fitz

pdf_path = "data-reports/weather_report_sample.pdf"
doc = fitz.open(pdf_path)

print("Total pages:", len(doc))

Total pages: 1


### 3.2 Extract all text and visualize along with y coordinate

In [20]:
text_blocks = []

for page_num, page in enumerate(doc):
    blocks = page.get_text("blocks")
    
    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block
        
        if block_type == 0 and text.strip():
            text_blocks.append({
                "type": "text",
                "content": text.strip(),
                "page": page_num + 1,
                "y": y0
            })

# Sort and display
text_blocks_sorted = sorted(text_blocks, key=lambda x: (x["page"], x["y"]))

for block in text_blocks_sorted:
    print(f"Page {block['page']} | Y={block['y']:.2f}")
    print(block["content"])
    print("-" * 50)

Page 1 | Y=76.74
Weekly Weather Report
--------------------------------------------------
Page 1 | Y=126.85
This weather report provides an overview of the weekly temperature trends along with a summary
of atmospheric conditions. The data below illustrates daily high temperatures and key weather
metrics for the week.
--------------------------------------------------
Page 1 | Y=422.05
The bar chart above shows a gradual increase in temperature toward the end of the week, with
Thursday recording the highest temperature. Overall, conditions remained warm and stable.
--------------------------------------------------
Page 1 | Y=470.65
Day
High (°C)
Condition
Humidity (%)
--------------------------------------------------
Page 1 | Y=488.65
Mon
30
Sunny
45
--------------------------------------------------
Page 1 | Y=506.65
Tue
32
Partly Cloudy
50
--------------------------------------------------
Page 1 | Y=524.65
Wed
29
Cloudy
60
--------------------------------------------------
Page 1 |

### 3.3 Extract all images and visualize along with y coordinate.

In [21]:
image_blocks = []

for page_num, page in enumerate(doc):
    images = page.get_images(full=True)
    
    for img in images:
        bbox = page.get_image_bbox(img)
        
        image_blocks.append({
            "type": "image",
            "content": "[IMAGE]",
            "page": page_num + 1,
            "y": bbox.y0
        })

# Sort and display
image_blocks_sorted = sorted(image_blocks, key=lambda x: (x["page"], x["y"]))

for block in image_blocks_sorted:
    print(f"Page {block['page']} | Y={block['y']:.2f} | IMAGE")

Page 1 | Y=185.20 | IMAGE


### 3.4 Extract all tables and visualize along with y coordinate

In [22]:
# For tables we are using pdfplumber

import pdfplumber

table_blocks = []

with pdfplumber.open(pdf_path) as plumber_doc:
    for page_num, page in enumerate(plumber_doc.pages):
        tables = page.find_tables()
        
        for table in tables:
            table_blocks.append({
                "type": "table",
                "content": "[TABLE]",
                "page": page_num + 1,
                "y": table.bbox[1]
            })

# Sort and display
table_blocks_sorted = sorted(table_blocks, key=lambda x: (x["page"], x["y"]))

for block in table_blocks_sorted:
    print(f"Page {block['page']} | Y={block['y']:.2f} | TABLE")

Page 1 | Y=468.40 | TABLE


### 3.5 Let's reconstruct and visualize. We will have a text-table duplication problem here

In [23]:
all_blocks = text_blocks + image_blocks + table_blocks

reconstructed = sorted(all_blocks, key=lambda x: (x["page"], x["y"]))

for block in reconstructed:
    print(f"\nPage {block['page']} | {block['type'].upper()} | Y={block['y']:.2f}")
    print(block["content"])


Page 1 | TEXT | Y=76.74
Weekly Weather Report

Page 1 | TEXT | Y=126.85
This weather report provides an overview of the weekly temperature trends along with a summary
of atmospheric conditions. The data below illustrates daily high temperatures and key weather
metrics for the week.

Page 1 | IMAGE | Y=185.20
[IMAGE]

Page 1 | TEXT | Y=422.05
The bar chart above shows a gradual increase in temperature toward the end of the week, with
Thursday recording the highest temperature. Overall, conditions remained warm and stable.

Page 1 | TABLE | Y=468.40
[TABLE]

Page 1 | TEXT | Y=470.65
Day
High (°C)
Condition
Humidity (%)

Page 1 | TEXT | Y=488.65
Mon
30
Sunny
45

Page 1 | TEXT | Y=506.65
Tue
32
Partly Cloudy
50

Page 1 | TEXT | Y=524.65
Wed
29
Cloudy
60

Page 1 | TEXT | Y=542.65
Thu
35
Sunny
40

Page 1 | TEXT | Y=560.65
Fri
33
Clear
42

Page 1 | TEXT | Y=597.25
In summary, the week experienced predominantly sunny weather with moderate humidity levels.
Such patterns are typical during stab

- Table contents are duplicated because both text and table extractors extract them.

## 4. How to overcome Table and Text Overlap?

### Step 1: Collect Table Bounding Boxes First

In [24]:
import pdfplumber

table_bboxes_per_page = {}

with pdfplumber.open(pdf_path) as plumber_doc:
    for page_num, page in enumerate(plumber_doc.pages):
        tables = page.find_tables()
        table_bboxes_per_page[page_num] = [t.bbox for t in tables]

### Step 2: Extract Text BUT Skip Table Regions

In [25]:
text_blocks = []

for page_num, page in enumerate(doc):
    blocks = page.get_text("blocks")
    table_bboxes = table_bboxes_per_page.get(page_num, [])
    
    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block
        
        if block_type != 0 or not text.strip():
            continue
        
        # Check if block is inside any table bbox
        inside_table = False
        for tb in table_bboxes:
            if x0 >= tb[0] and y0 >= tb[1] and x1 <= tb[2] and y1 <= tb[3]:
                inside_table = True
                break
        
        if not inside_table:
            text_blocks.append({
                "type": "text",
                "content": text.strip(),
                "page": page_num + 1,
                "y": y0
            })

### Final Reconstruction

In [26]:
all_blocks = text_blocks + image_blocks + table_blocks
reconstructed = sorted(all_blocks, key=lambda x: (x["page"], x["y"]))

for block in reconstructed:
    print(f"\nPage {block['page']} | {block['type'].upper()} | Y={block['y']:.2f}")
    print(block["content"])


Page 1 | TEXT | Y=76.74
Weekly Weather Report

Page 1 | TEXT | Y=126.85
This weather report provides an overview of the weekly temperature trends along with a summary
of atmospheric conditions. The data below illustrates daily high temperatures and key weather
metrics for the week.

Page 1 | IMAGE | Y=185.20
[IMAGE]

Page 1 | TEXT | Y=422.05
The bar chart above shows a gradual increase in temperature toward the end of the week, with
Thursday recording the highest temperature. Overall, conditions remained warm and stable.

Page 1 | TABLE | Y=468.40
[TABLE]

Page 1 | TEXT | Y=597.25
In summary, the week experienced predominantly sunny weather with moderate humidity levels.
Such patterns are typical during stable high-pressure systems. This concludes the one-page
sample weather report.


### Now, reconstructed without any duplicated values